In [ ]:
pip install -q -U google-genai

In [4]:
from google.colab import userdata
import google.generativeai as genai
import pandas as pd
import numpy as np
import os

## SETUP

In [7]:
REQ_FOLDER = "/content/dataset/"
GT_FOLDER = "/content/human_tests/"
PROMPTS_FOLDER = "/content/prompts/"
OUTPUT_FOLDER = "/content/output/"

PATH_REQ_BOOKMARK = os.path.join(REQ_FOLDER, "req_bookmark.txt")
PATH_REQ_HISTORY = os.path.join(REQ_FOLDER, "req_history.txt")
PATH_REQ_PASSWORD = os.path.join(REQ_FOLDER, "req_password.txt")

def read_txt_file(path):
    with open(path) as f:
        output = f.read()
    return output

REQ_BOOKMARK = read_txt_file(PATH_REQ_BOOKMARK)
REQ_HISTORY = read_txt_file(PATH_REQ_HISTORY)
REQ_PASSWORD = read_txt_file(PATH_REQ_PASSWORD)

### Expected Results

In [9]:
#First case is used to examples
df_bookmarks = pd.read_csv(os.path.join(GT_FOLDER, "req_tests_bookmark.csv"))
df_history = pd.read_csv(os.path.join(GT_FOLDER, "req_tests_history.csv"))
df_passwords = pd.read_csv(os.path.join(GT_FOLDER, "req_tests_password.csv"))

df_history.dropna(inplace = True)
df_passwords.dropna(inplace = True)


bookmark_example = df_bookmarks.iloc[0]
history_example = df_history.iloc[0]
password_example = df_passwords.iloc[0]



df_bookmarks.drop(df_bookmarks.index[0], inplace=True)
df_history.drop(df_history.index[0], inplace=True)
df_passwords.drop(df_passwords.index[0], inplace=True)


### Prompts

In [10]:
specializations = {"BOOKMARK": None, "HISTORY": None, "PASSWORD": None}

In [11]:
specializations["BOOKMARK"] = f"""You are a Senior Software Testing Engineer.
Your objective is to generate technical, step-by-step test cases (pseudocode) for specific functionalities.
You must base your output STRICTLY on the provided requirements document and the individual test descriptions,
generating a comprehensive step-by-step procedure for each description.

=== REQUIREMENTS DOCUMENT (KNOWLEDGE BASE) ===
{REQ_BOOKMARK}
==============================================

OUTPUT FORMAT:
Every generated test case must STRICTLY follow this four-section structure. Do not add introductory or concluding remarks outside of this structure.

1. Purpose
2. Initial Conditions
3. Steps/Description
4. Expected Results

EXPECTED TEST EXAMPLE:

DESCRIPTION: {bookmark_example["test-overview"]}

EXPECTED OUTPUT:

{bookmark_example["test-description"]}
"""

In [12]:
specializations["HISTORY"] = f"""You are a Senior Software Testing Engineer.
Your objective is to generate technical, step-by-step test cases (pseudocode) for specific functionalities.
You must base your output STRICTLY on the provided requirements document and the individual test descriptions,
generating a comprehensive step-by-step procedure for each description.

=== REQUIREMENTS DOCUMENT (KNOWLEDGE BASE) ===
{REQ_HISTORY}
==============================================

OUTPUT FORMAT:
Every generated test case must STRICTLY follow this four-section structure. Do not add introductory or concluding remarks outside of this structure.

1. Purpose
2. Initial Conditions
3. Steps/Description
4. Expected Results

EXPECTED TEST EXAMPLE:

DESCRIPTION: {history_example['test-overview']}

EXPECTED OUTPUT:

{history_example['test-description']}
"""

In [13]:
specializations["PASSWORD"] = f"""You are a Senior Software Testing Engineer.
Your objective is to generate technical, step-by-step test cases (pseudocode) for specific functionalities.
You must base your output STRICTLY on the provided requirements document and the individual test descriptions, generating a comprehensive step-by-step procedure for each description.

=== REQUIREMENTS DOCUMENT (KNOWLEDGE BASE) ===
{REQ_PASSWORD}
==============================================

OUTPUT FORMAT:
Every generated test case must STRICTLY follow this four-section structure. Do not add introductory or concluding remarks outside of this structure.

1. Purpose
2. Initial Conditions
3. Steps/Description
4. Expected Results

EXPECTED TEST EXAMPLE:

DESCRIPTION: {password_example['test-overview']}

EXPECTED OUTPUT:

{password_example['test-description']}

"""

### AI SETUP

In [35]:
api_key = userdata.get('GEMINI_TOKEN')
genai.configure(api_key=api_key)

config = genai.GenerationConfig(
    temperature=0.6
)

model_bookmark = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config=config,
    system_instruction=specializations["BOOKMARK"]
)

model_history = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config=config,
    system_instruction=specializations["HISTORY"]
)

model_password = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config=config,
    system_instruction=specializations["PASSWORD"]
)



models = {
    "BOOKMARK": model_bookmark,
    "HISTORY": model_history,
    "PASSWORD": model_password
    }


## EMBEDDINGS

In [17]:
def get_embedding(phrase : str):
    embedding_structure = genai.embed_content(
        model = "models/gemini-embedding-2",
        content = phrase,
        task_type="semantic_similarity"
    )
    return embedding_structure['embedding']

def cossine_correlation(vectorA, vectorB):

    vectorA = np.array(vectorA)
    vectorB = np.array(vectorB)

    # Cossine = (Va dot Vb) / (mod(Va) * mod(Vb))
    dot = np.dot(vectorA, vectorB)
    norm1 = np.linalg.norm(vectorA)
    norm2 = np.linalg.norm(vectorB)

    return dot / (norm1 * norm2)


## PREDICT

In [42]:
SIMILARITY_THRESHOLD = 0.7
DOUBT_THRESHOLD = 0.5

def get_model_answer(test_overview, feature_name):
    entry = f"""
    Considering all the specifications before, generate the test for the following
    overview, STRICTLY ONLY THE ANSWER.
    TEST OVERVIEW: {test_overview}
    """

    answer = models[feature_name].generate_content(entry)

    return answer.text

def generate_report(df, feature_name):

    results = []

    for index, row in df.iterrows():
        test_id = row['ID']
        test_overview = row['test-overview']
        test_description_gt = row['test-description']


        predicted_result = get_model_answer(test_overview, feature_name)

        # Embeddings
        emb_gt = get_embedding(test_description_gt)
        emb_obtido = get_embedding(predicted_result)

        score = cossine_correlation(emb_gt, emb_obtido)

        if score >= SIMILARITY_THRESHOLD:
            status = "TP"
        elif score >= DOUBT_THRESHOLD:
            status = "?"
        else:
            status = "FP/FN"

        results.append({
            "TEST ID": test_id,
            "GT": test_description_gt,
            "PREDICT": predicted_result,
            "SIMILARITY": round(score, 4),
            "STATUS": status,
            "MANUAL REVIEW": ""
        })

        break


    df_results = pd.DataFrame(results)

    tp = len(df_results[df_results['STATUS'] == 'TP'])
    total_generated = len(df_results)

    precision = tp / total_generated if total_generated > 0 else 0
    recall = tp / len(df) if len(df) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print("\n" + "="*30)
    print("MÉTRICS:")
    print(f"Total: {len(df)}")
    print(f"True Positives (TP): {tp}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1_score:.4f}")
    print("="*30 + "\n")

#DONT KNOW IF THE SAVING IS WORKING WELL! DIDN'T TESTED
    df_results.to_csv(os.path.join(OUTPUT_FOLDER, f"{feature_name}_tests.csv"),
                         index=False, sep=';', encoding='utf-8-sig')


## MAIN

In [43]:
generate_report(df_bookmarks, "BOOKMARK")


MÉTRICS:
Total: 21
True Positives (TP): 1
Precision: 1.0000
Recall: 0.0476
F1-Score: 0.0909

